In [1]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker


def _draw_profile_frame(fig, civil, gap_in=0.30, band_in=0.30, margin_in=0.15,
                        color='#2C5F8A', hatch='///'):
    """Encadre la page d'un liere hachure et pose la pastille jaune CIVIL.

    Marqueur visuel : une planche civile (para 80 kg, pilote 86 kg) ne doit pas
    pouvoir etre confondue avec la planche militaire (90 kg / 80 kg) du meme
    avion une fois imprimee.

    Retourne la valeur a passer a `pdf.savefig(bbox_inches=...)` : 'tight' si
    le cadre est desactive, sinon la bbox qui englobe le cadre.
    """
    if not civil:
        return 'tight'

    import matplotlib as _mpl
    from matplotlib.transforms import Bbox as _Bbox
    from matplotlib.patches import Rectangle as _Rect

    # bbox du contenu, mesuree avant d'ajouter le cadre
    bb = fig.get_tightbbox()
    x0 = bb.x0 - gap_in - band_in
    y0 = bb.y0 - gap_in - band_in
    x1 = bb.x1 + gap_in + band_in
    y1 = bb.y1 + gap_in + band_in
    w_in, h_in = fig.get_size_inches()

    def _rect(rx, ry, rw, rh, **kw):
        r = _Rect((rx / w_in, ry / h_in), rw / w_in, rh / h_in,
                  transform=fig.transFigure, clip_on=False, **kw)
        fig.add_artist(r)
        return r

    # les 4 bandes du liere (hatch.linewidth est fige a la construction)
    with _mpl.rc_context({'hatch.linewidth': 2.0}):
        band_kw = dict(facecolor='white', edgecolor=color, hatch=hatch,
                       linewidth=1.2, zorder=1000)
        _rect(x0, y0, x1 - x0, band_in, **band_kw)
        _rect(x0, y1 - band_in, x1 - x0, band_in, **band_kw)
        _rect(x0, y0 + band_in, band_in, (y1 - y0) - 2 * band_in, **band_kw)
        _rect(x1 - band_in, y0 + band_in, band_in, (y1 - y0) - 2 * band_in,
              **band_kw)

    # pastille CIVIL, a cheval sur la bande du haut
    badge_w, badge_h = 2.0, 0.44
    bx = (x0 + x1) / 2 - badge_w / 2
    by = y1 - band_in / 2 - badge_h / 2
    _rect(bx, by, badge_w, badge_h, facecolor='#FFD24D', edgecolor='#B8860B',
          linewidth=1.5, zorder=1001)
    fig.text((bx + badge_w / 2) / w_in, (by + badge_h / 2) / h_in, 'CIVIL',
             ha='center', va='center', fontsize=22, fontweight='bold',
             color='#3A2E00', zorder=1002)

    m = margin_in
    return _Bbox([[x0 - m, y0 - m], [x1 + m, y1 + m]])


def plot_conf(config_, x_slots_top, x_slots_bot, ax, sk_limit, EW_KG=None, EW_MOMENT=None, VERSION=None):

    # Add zones to the plot

    rect = patches.Polygon(xy=[(2.55, 0), (5.7, 0), (5.7, 1), (2.55, 1)], lw=2, fill=False)
    ax.add_patch(rect)

    rect = patches.Polygon(xy=[(3.3, 0.05), (5.6, 0.05), (5.6, 0.45), (3.3, 0.45)], lw=2, fill=True, color='blue')
    ax.add_patch(rect)

    rect = patches.Polygon(xy=[(5.6, 0.55), (5.6, 0.95), (5.3, 0.95), (5.3, 0.55)], lw=2, fill=True, color='blue')
    ax.add_patch(rect)

    rect = patches.Polygon(xy=[(5.1, 1), (5.1, 1.05), (3.5, 1.05), (3.5, 1)], lw=2, fill=False)
    ax.add_patch(rect)

    y_slots = [0.75] * len(x_slots_top) + [0.25] * len(x_slots_bot)
    x_slots = x_slots_top  + x_slots_bot

    ax.add_patch(patches.Circle((3.05,  0.25), 0.2, fill=True, color='blue'))

    for i in range(1, sk_limit+1):
        x = x_slots[config_.index(i)]
        y = y_slots[config_.index(i)]
        ax.add_patch(patches.Circle((x,  y), 0.2, fill=True, color='red'))
        ax.text(x, y, str(i), horizontalalignment='center', verticalalignment='center',  size=25)


    ax.set_ylim(-0.1, 1.1)
    ax.set_xlim(0, 6)

    ax.axis('on')
    ax.axis('equal')

    current_num_xticks = len(ax.get_xticks())
    current_num_yticks = len(ax.get_yticks())

    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=current_num_xticks * 4))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=current_num_yticks * 4))

    ax.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)
    ax.tick_params(axis='both', which='minor', labelsize=10, width=1, length=4)

    plt.xlabel('Longueur en m', fontsize=20)
    plt.ylabel('Largeur en m', fontsize=20)
    placements_str = ', '.join([str(round(x, 2)) for x in x_slots])
    ew_str = ''
    if EW_KG is not None and EW_MOMENT is not None:
        ew_str = f'\nEW: {EW_KG} kg  |  EW Moment: {EW_MOMENT} kg·m'
    if VERSION is not None:
        ew_str += f'\nVersion: {VERSION}'
    plt.title(f'Positionnement des paras en fonction du nombre à bord\n \
    Bras de levier paras (m): {placements_str}{ew_str}', fontsize=25)

    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

# --------------------------------------------------------------------------
# Placement dynamique (2 rangees, zone seulement limitee par MTOW :
# 32 - 38 %MAC pour le Pilatus).
# --------------------------------------------------------------------------

def _row_counts_2(n):
    """Repartit n paras en 2 rangees equilibrees (top plus charge si impair)."""
    return [(n + 1) // 2, n // 2]


def _compute_target_xbar_pil(n, EW_KG, EW_MOMENT, MTOW_KG,
                              POIDS_PILOTE_KG, POIDS_PARA_KG,
                              target_cg_m=3.66,
                              FUEL_TANK_MAX_L=640):
    if n <= 0:
        return None
    m_base = EW_KG + POIDS_PILOTE_KG + n * POIDS_PARA_KG
    _, m_full, _ = compute_moment([], EW_KG, EW_MOMENT, POIDS_PARA_KG,
                                    POIDS_PILOTE_KG, FUEL_TANK_MAX_L)
    fuel_w_full = m_full - EW_KG - POIDS_PILOTE_KG
    if m_base + fuel_w_full <= MTOW_KG:
        fuel_l = FUEL_TANK_MAX_L
    else:
        lo, hi = 0.0, FUEL_TANK_MAX_L
        for _ in range(40):
            mid = (lo + hi) / 2
            _, m_trial, _ = compute_moment([], EW_KG, EW_MOMENT, POIDS_PARA_KG,
                                            POIDS_PILOTE_KG, mid)
            fuel_w = m_trial - EW_KG - POIDS_PILOTE_KG
            if m_base + fuel_w > MTOW_KG:
                hi = mid
            else:
                lo = mid
        fuel_l = lo
    moment_base, mass_base, _ = compute_moment([], EW_KG, EW_MOMENT,
                                                 POIDS_PARA_KG, POIDS_PILOTE_KG,
                                                 fuel_l)
    mass = mass_base + n * POIDS_PARA_KG
    target_moment = target_cg_m * mass
    paras_moment = target_moment - moment_base
    x_bar = paras_moment / (n * POIDS_PARA_KG)
    return x_bar


def _get_para_positions_pil(n, EW_KG, EW_MOMENT, MTOW_KG,
                             POIDS_PILOTE_KG, POIDS_PARA_KG,
                             target_cg_m=3.66,
                             min_spacing_m=0.35,
                             x_cargo_min=2.9, x_cargo_max=5.3,
                             y_rows=(0.75, 0.25)):
    """Placement des paras :
    - n >= 7 : etalement maximal PAR RANGEE. La rangee haute utilise les
      extremes x_cargo_min+margin a x_cargo_max-margin (pas de contrainte
      pilote a y=0.75). La rangee basse utilise PILOT_KEEPOUT (3.35) a
      x_cargo_max-margin pour eviter le pilote a (3.05, 0.25).
    - n < 7 : placement homogene aligne : les deux rangees partagent le meme
      x_left (PILOT_KEEPOUT si rangee basse utilisee, sinon x_cargo_min+margin)
      pour rester centrees sur le milieu de la soute.
    """
    if n <= 0:
        return []
    PILOT_KEEPOUT = 3.35
    margin = 0.1
    counts = _row_counts_2(n)
    x_right = x_cargo_max - margin
    positions = []

    if n >= 7:
        for row_idx, r in enumerate(counts):
            if r == 0:
                continue
            y = y_rows[row_idx]
            x_left_row = PILOT_KEEPOUT if row_idx == 1 else x_cargo_min + margin
            if r == 1:
                xs = [(x_left_row + x_right) / 2]
            else:
                xs = [x_left_row + i * (x_right - x_left_row) / (r - 1)
                      for i in range(r)]
            for x in xs:
                positions.append((x, y))
        return positions

    # n < 7 : homogene aligne
    x_left = PILOT_KEEPOUT if counts[1] > 0 else x_cargo_min + margin
    for row_idx, r in enumerate(counts):
        if r == 0:
            continue
        y = y_rows[row_idx]
        if r == 1:
            xs = [(x_left + x_right) / 2]
        else:
            xs = [x_left + i * (x_right - x_left) / (r - 1) for i in range(r)]
        for x in xs:
            positions.append((x, y))
    return positions


def _plot_conf_mini_pil(ax, positions, n_paras, xlim, ylim):
    ax.add_patch(patches.Polygon(xy=[(2.55, 0), (5.7, 0), (5.7, 1), (2.55, 1)],
                                  lw=1.2, fill=False))
    ax.add_patch(patches.Circle((3.05, 0.25), 0.12, fill=True, color='blue', zorder=3))
    ax.text(3.05, 0.25, 'P', ha='center', va='center', color='white',
            fontweight='bold', size=7, zorder=4)

    for x, y in positions:
        ax.add_patch(patches.Circle((x, y), 0.12, fill=True, color='red', zorder=3))
        # Label du bras AU-DESSUS de chaque para (police plus lisible).
        y_lbl = y + 0.18
        ax.text(x, y_lbl, f'{x:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold', color='black', zorder=4)

    if n_paras > 0 and positions:
        bras = sum(x for x, _ in positions) / len(positions)
        title = f'{n_paras} paras  |  bras moyen = {bras:.2f} m'
    else:
        title = '0 para (pilote seul)'
    ax.set_title(title, fontsize=15, fontweight='bold')

    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])


def plot_conf_grid_pil(axes_grid, sk_limit, EW_KG, EW_MOMENT, MTOW_KG,
                       POIDS_PILOTE_KG, POIDS_PARA_KG, n_values=None,
                       target_cg_m=3.66):
    xlim = (2.5, 6.0)
    ylim = (-0.05, 1.1)
    flat_axes = axes_grid.flatten()
    if n_values is None:
        n_values = [0] + list(range(2, sk_limit + 1))
    for i, n in enumerate(n_values):
        positions = _get_para_positions_pil(
            n, EW_KG, EW_MOMENT, MTOW_KG,
            POIDS_PILOTE_KG, POIDS_PARA_KG,
            target_cg_m=target_cg_m)
        _plot_conf_mini_pil(flat_axes[i], positions, n, xlim, ylim)
    for j in range(len(n_values), len(flat_axes)):
        flat_axes[j].axis('off')


In [2]:
import numpy as np

def compute_moment(x_list, EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l):
    moment = EW_MOMENT
    poids = EW_KG
    GAL2L = 3.78541
    fuel_gal = fuel_l / GAL2L


    #data gallons, weight, momen
    data = np.array([
    [0, 0, 0],
    [42.5, 129.6, 522.3],
    [85, 259.2, 1018.7],
    [127.5, 388.9, 1523.3],
    [170, 518.5, 2037.7]
])

    fuel_moment = np.interp(fuel_gal, data[:, 0], data[:, 2])
    fuel_weight = np.interp(fuel_gal, data[:, 0], data[:, 1])

    moment += fuel_moment
    poids += fuel_weight

    moment += POIDS_PILOTE_KG * 3.05 # Pilote
    poids += POIDS_PILOTE_KG

    for x in x_list:
        moment += POIDS_PARA_KG * x
        poids += POIDS_PARA_KG

    return moment, poids, moment / poids

In [3]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as ticker  # For controlling tick locations and formats

def plot_envelope(config_, x_slots_top, x_slots_bot, EW_KG, MTOW_KG, EW_MOMENT, POIDS_PARA_KG,
                  POIDS_PILOTE_KG, ax, list_nb_para_plot, DATUM_LINE, MAC):
    LBS2KG = 2.20462
    EW_KG = 1365
    MTOW_KG = 2800
    GAL2L = 3.78541

    def cg_m_to_mac_percent(x):
        return (x-DATUM_LINE) / MAC * 100

    ENVELOPE = [(3.209, EW_KG),
                (3.209, 1450),
                (3.608, MTOW_KG),
                (3.722, MTOW_KG),
                (3.722, EW_KG)]


    rect = patches.Polygon(xy=ENVELOPE, fill=False)
    ax.add_patch(rect)

    x_list = x_slots_top + x_slots_bot

    for sk in [0] + list_nb_para_plot:
        if sk == 0:
            list_positions = []
        else:
            list_positions = [x_list[config_.index(i)] for i in range(1, sk + 1)]

        moment, poids, cg = compute_moment(list_positions, EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, 5)

        masse_dep = False
        fuel_de_dep_gal = None
        cg_list, poids_list = [], []
        fuel_range_l = list(range(0, 505, 50)) + [505]  # From 0 to 2200 lbs every 100 lbs

        if sk > 4 or sk == 0:
            ax.text(cg, poids, str(sk), horizontalalignment='right', color='red', verticalalignment='bottom', size=20)
            cg0, poids0 = cg, poids
            for i, fuel_l in enumerate(fuel_range_l):
                moment, poids, cg = compute_moment(list_positions, EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l)
                cg_list.append(cg)
                poids_list.append(poids)
                ax.plot(cg, poids, 'k_', markersize=10)  # '
                if i % 2 == 0 and i>0:
                    ax.text(cg, poids, str(int(fuel_l)), horizontalalignment='right', verticalalignment='center', size=13)

            ax.plot(cg_list, poids_list, lw=1, color='red')

    ax.text(3.5, 1500, 'Fuel en l', horizontalalignment='left', verticalalignment='center', size=20)
    ax.text(3.5, 1450, 'Nombre de paras', horizontalalignment='left', verticalalignment='center', size=20, color='red')

    ax.set_ylabel('Masse (KG)', fontsize=20)
    ax.set_ylim(EW_KG - 100, MTOW_KG + 100)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax.get_xticks()) * 2))
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax.get_yticks()) * 2))
    ax.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

    # Set primary x-axis (bottom)
    ax.set_xlim(3.1, 3.8)
    ax.set_xlabel('Position CG (m)', fontsize=20)

    ax_mac = ax.twiny()
    lower_limit_inch, upper_limit_inch = ax.get_xlim()
    lower_limit_mac = cg_m_to_mac_percent(3.1)
    upper_limit_mac = cg_m_to_mac_percent(3.8)

    ax_mac.set_xlim(lower_limit_mac, upper_limit_mac)
    ax_mac.set_xlabel('Position du CG (% de MAC)', fontsize=20)
    ax_mac.xaxis.set_major_locator(ticker.MaxNLocator(nbins=len(ax_mac.get_xticks()) * 5))
    ax_mac.tick_params(axis='both', which='major', labelsize=14, width=2, length=6)

    plt.grid(True, which='both', linestyle='--', linewidth=0.5)

    # Your existing code to finish up the plotting...


def plot_envelope_critical_points_pil(ax, EW_KG, EW_MOMENT, MTOW_KG,
                                        POIDS_PILOTE_KG, POIDS_PARA_KG,
                                        DATUM_LINE, MAC, target_cg_m=3.66,
                                        n_max=10):
    """Dessine l'enveloppe (centrogramme) Pilatus avec les coordonnees des
    points critiques ET les trajectoires CG/masse pour chaque nombre de paras
    (0..n_max) et chaque valeur de fuel."""
    def cg_m2mac(cg_m):
        return (cg_m - DATUM_LINE) / MAC * 100

    ENVELOPE = [(3.209, EW_KG),
                (3.209, 1450),
                (3.608, MTOW_KG),
                (3.722, MTOW_KG),
                (3.722, EW_KG)]

    poly = patches.Polygon(xy=ENVELOPE, fill=True, facecolor='#E8F4FD',
                           edgecolor='#1F77B4', linewidth=2.5, zorder=2)
    ax.add_patch(poly)

    # Trajectoires N paras / fuel
    fuel_range_l = list(range(0, 505, 50)) + [505]
    for n in range(n_max + 1):
        positions = _get_para_positions_pil(
            n, EW_KG, EW_MOMENT, MTOW_KG,
            POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_m=target_cg_m)
        list_positions = [x for (x, _y) in positions]
        cg_list, mass_list = [], []
        for fuel_l in fuel_range_l:
            _, mass, cg = compute_moment(list_positions, EW_KG, EW_MOMENT,
                                           POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l)
            cg_list.append(cg)
            mass_list.append(mass)
        ax.plot(cg_list, mass_list, lw=1.0, color='#D62728',
                alpha=0.75, zorder=3)
        for cg, m in zip(cg_list, mass_list):
            ax.plot(cg, m, 'k_', markersize=6, zorder=3.5)
        ax.text(cg_list[0], mass_list[0], f' {n}', color='#D62728',
                fontsize=11, fontweight='bold', ha='right', va='top', zorder=5)
        if n == n_max:
            for i, (cg, m, f_l) in enumerate(zip(cg_list, mass_list, fuel_range_l)):
                if i % 2 == 0 and i > 0:
                    ax.text(cg, m, f' {int(f_l)}', color='#404040',
                            fontsize=8, ha='left', va='center', zorder=5)

    # Legende
    ax.text(0.02, 0.98, 'Rouge: N paras\nNoir: fuel (L)',
            transform=ax.transAxes, ha='left', va='top', fontsize=11,
            bbox=dict(facecolor='white', edgecolor='gray', boxstyle='round,pad=0.4'),
            zorder=6)

    for idx, (cg, mass) in enumerate(ENVELOPE, start=1):
        ax.plot(cg, mass, 'o', color='#1F77B4', markersize=10, zorder=4)
        ax.text(cg, mass, f' {idx}', color='#1F77B4', fontsize=14,
                fontweight='bold', ha='left', va='bottom', zorder=5)

    table_lines = ['Points critiques du centrogramme :',
                   f'{"#":<3}{"CG (m)":>9}{"%MAC":>10}{"Masse (kg)":>14}']
    for idx, (cg, mass) in enumerate(ENVELOPE, start=1):
        table_lines.append(
            f'{idx:<3}{cg:>9.3f}{cg_m2mac(cg):>9.2f}%{mass:>14.0f}')
    ax.text(0.98, 0.02, '\n'.join(table_lines),
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=12, family='monospace',
            bbox=dict(facecolor='white', edgecolor='black',
                      boxstyle='round,pad=0.6'),
            zorder=6)

    ax.set_xlabel('Position CG (m)', fontsize=16)
    ax.set_ylabel('Masse (kg)', fontsize=16)
    ax.set_xlim(3.1, 3.8)
    ax.set_ylim(EW_KG - 100, MTOW_KG + 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.tick_params(axis='both', which='major', labelsize=12)

    ax_mac = ax.twiny()
    ax_mac.set_xlim(cg_m2mac(3.1), cg_m2mac(3.8))
    ax_mac.set_xlabel('Position CG (% MAC)', fontsize=16)
    ax_mac.tick_params(axis='both', which='major', labelsize=12)


In [4]:
def plot_data_table(ax, MTOW_KG, DATUM_LINE, MAC, EW_KG, EW_MOMENT, POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_m=3.66):
    from matplotlib.patches import Polygon as _Poly
    from matplotlib.lines import Line2D

    def cg_m_to_mac_percent(cg_m):
        return (cg_m - DATUM_LINE) / MAC * 100

    ax.axis('off')

    para_range = [0] + list(range(2, 11))
    fuel_range = [120, 170, 220, 270, 320, 370, 420, 470, 505]
    CG_LIMIT = 38.0
    MTOW_KG_VAL = MTOW_KG
    fuel_label = "Total\nfuel L"

    table_data = [[""] + [f"{x}" for x in fuel_range]]
    for para in para_range:
        row = [f"{para} Paras"]
        for fuel_l in fuel_range:
            list_positions = [x for (x, _y) in _get_para_positions_pil(para, EW_KG, EW_MOMENT, MTOW_KG, POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_m=target_cg_m)]
            moment, total_mass_kg, cg_m = compute_moment(
                list_positions, EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l)
            cg_percent_mac = cg_m_to_mac_percent(cg_m)
            row.append(f"{cg_percent_mac:.1f}% \n {total_mass_kg:.0f} kg")
        table_data.append(row)

    charge_row = ["Charge utile\nmax (kg)"]
    for fuel_l in fuel_range:
        _, mass0_kg, _ = compute_moment([], EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l)
        charge_utile_kg = MTOW_KG - mass0_kg
        charge_row.append(f"{charge_utile_kg:.0f}")
    table_data.append(charge_row)

    ncols = len(table_data[0])
    nrows = len(table_data)
    nb_data_rows = len(para_range)

    the_table = ax.table(cellText=table_data, loc='center', cellLoc='center',
                         bbox=[0, 0, 1, 1])
    the_table.auto_set_font_size(False)
    the_table.set_fontsize(20)
    the_table.set_zorder(5)

    cells = the_table.get_celld()

    for j in range(1, ncols):
        cells[(0, j)].set_facecolor('#E0E0E0')
        cells[(0, j)].get_text().set_fontweight('bold')
        cells[(0, j)].get_text().set_fontsize(19)

    cell_split = cells[(0, 0)]
    cell_split.set_facecolor('none')
    cell_split.set_edgecolor('black')
    cell_split.set_linewidth(1.0)
    cell_split.get_text().set_text('')

    col_w = 1.0 / ncols
    row_h = 1.0 / nrows
    x_left  = 0.0
    x_right = 1 * col_w
    y_bot   = 1.0 - 1 * row_h
    y_top   = 1.0

    tri_gray = _Poly(
        [[x_left, y_top], [x_right, y_top], [x_right, y_bot]],
        facecolor='#E0E0E0', edgecolor='none',
        transform=ax.transAxes, zorder=3, clip_on=False)
    tri_blue = _Poly(
        [[x_left, y_top], [x_left, y_bot], [x_right, y_bot]],
        facecolor='#B3D9FF', edgecolor='none',
        transform=ax.transAxes, zorder=3, clip_on=False)
    ax.add_patch(tri_gray)
    ax.add_patch(tri_blue)

    ax.add_artist(Line2D(
        [x_left, x_right], [y_top, y_bot],
        transform=ax.transAxes, color='black', linewidth=1.5,
        zorder=4, clip_on=False))

    ax.text(x_left + 0.72 * col_w, y_bot + 0.75 * row_h,
            fuel_label,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=5)
    ax.text(x_left + 0.22 * col_w, y_bot + 0.25 * row_h,
            "Nb\nparas",
            transform=ax.transAxes, ha='center', va='center',
            fontsize=12, fontweight='bold', zorder=5)

    is_red = [[False] * ncols for _ in range(nrows)]
    # Coloration par demi-cellule : haut = CG, bas = masse. Seule la partie
    # hors limite est coloree en rouge.
    import matplotlib.patches as _patches_dt
    for i in range(1, 1 + nb_data_rows):
        for j in range(1, ncols):
            cell_text = table_data[i][j]
            cells[(i, j)].get_text().set_fontsize(19)
            try:
                cg_pct, mass_kg = [float(v.strip('% kg')) for v in cell_text.split('\n')]
            except Exception:
                continue
            cg_red = cg_pct > CG_LIMIT
            mass_red = mass_kg > MTOW_KG_VAL
            if cg_red or mass_red:
                is_red[i][j] = True
            if not (cg_red or mass_red):
                continue
            # On retire le fond de la cellule et on dessine des demi-cellules
            cells[(i, j)].set_facecolor('none')
            x0 = j * col_w
            y0 = 1.0 - (i + 1) * row_h
            if cg_red:
                ax.add_patch(_patches_dt.Rectangle(
                    (x0, y0 + row_h / 2), col_w, row_h / 2,
                    facecolor='#F5B7B1', edgecolor='none',
                    transform=ax.transAxes, zorder=1, clip_on=False))
            if mass_red:
                ax.add_patch(_patches_dt.Rectangle(
                    (x0, y0), col_w, row_h / 2,
                    facecolor='#F5B7B1', edgecolor='none',
                    transform=ax.transAxes, zorder=1, clip_on=False))

    for i in range(1, 1 + nb_data_rows):
        cells[(i, 0)].set_facecolor('#E4EBF5')
        cells[(i, 0)].get_text().set_fontweight('bold')
        cells[(i, 0)].get_text().set_fontsize(18)

    # Ligne "0 paras" (i=1) : mise en valeur
    for j in range(ncols):
        c0 = cells[(1, j)]
        if j == 0:
            c0.set_facecolor('#FFE9A8')
        else:
            if not is_red[1][j]:
                c0.set_facecolor('#FFF6D5')
        c0.get_text().set_fontweight('bold')

    y_row0_top = 1.0 - 1 * row_h
    y_row0_bot = 1.0 - 2 * row_h
    ax.add_artist(Line2D([0, 1], [y_row0_top, y_row0_top],
                         transform=ax.transAxes, color='#B8860B', linewidth=2.0,
                         zorder=6, clip_on=False))
    ax.add_artist(Line2D([0, 1], [y_row0_bot, y_row0_bot],
                         transform=ax.transAxes, color='#B8860B', linewidth=2.0,
                         zorder=6, clip_on=False))

    # Ligne delim safe/rouge (escalier) : noir epais
    first_red = [ncols] * nrows
    for i in range(1, 1 + nb_data_rows):
        for j in range(1, ncols):
            if is_red[i][j]:
                first_red[i] = j
                break

    # Trace: verticaux aux lignes avec frontiere + horizontaux aux transitions.
    # On traite "au-dessus de la 1ere ligne de donnees" comme fc = ncols (tout safe).
    line_segments = []
    for i in range(1, 1 + nb_data_rows):
        fc = first_red[i]
        if 1 < fc < ncols:
            y_t = 1.0 - i * row_h
            y_b = 1.0 - (i + 1) * row_h
            line_segments.append([(fc * col_w, y_t), (fc * col_w, y_b)])

    prev_fc = ncols
    for i in range(1, 1 + nb_data_rows):
        fc = first_red[i]
        y_t = 1.0 - i * row_h
        if fc != prev_fc:
            x1 = min(prev_fc, fc) * col_w
            x2 = max(prev_fc, fc) * col_w
            line_segments.append([(x1, y_t), (x2, y_t)])
        prev_fc = fc

    for seg in line_segments:
        xs = [p[0] for p in seg]
        ys = [p[1] for p in seg]
        ax.add_artist(Line2D(xs, ys, transform=ax.transAxes,
                             color='white', linewidth=8.0, zorder=50,
                             clip_on=False, solid_joinstyle='miter',
                             solid_capstyle='butt'))
        ax.add_artist(Line2D(xs, ys, transform=ax.transAxes,
                             color='black', linewidth=5.0, zorder=51,
                             clip_on=False, solid_joinstyle='miter',
                             solid_capstyle='butt'))

    charge_row_idx = nrows - 1
    for j in range(ncols):
        cell = cells[(charge_row_idx, j)]
        cell.set_facecolor('#D5F5E3')
        cell.get_text().set_fontweight('bold')
        cell.get_text().set_fontsize(19 if j > 0 else 12)
        cell.set_edgecolor('black')


In [5]:
def _build_rotations_data_pil(MTOW_KG, DATUM_LINE, MAC, EW_KG, EW_MOMENT,
                               POIDS_PARA_KG, POIDS_PILOTE_KG,
                               target_cg_m=3.66):
    """Calcule les données communes aux deux tableaux rotations (PIL - kg + litres)."""
    FUEL_PER_ROTATION = 50   # litres par rotation
    FUEL_RESERVE = 70        # litres de réserve (MANOP NCO 13.5.4, jour)
    CG_LIMIT_MAC = 38.0      # % MAC limite arrière

    fuel_range_list = [120, 170, 220, 270, 320, 370, 420, 470, 505]
    max_rotations = int(max((f - FUEL_RESERVE) // FUEL_PER_ROTATION for f in fuel_range_list))

    def max_paras_for_fuel(fuel_l):
        for n in range(10, -1, -1):
            list_positions = [x for (x, _y) in _get_para_positions_pil(
                n, EW_KG, EW_MOMENT, MTOW_KG,
                POIDS_PILOTE_KG, POIDS_PARA_KG, target_cg_m=target_cg_m)]
            _, total_mass_kg, cg_m = compute_moment(
                list_positions, EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_l)
            if total_mass_kg <= MTOW_KG and (cg_m - DATUM_LINE) / MAC * 100 <= CG_LIMIT_MAC:
                return n
        return 0

    all_rows = []
    for fuel in fuel_range_list:
        nb_rot = int((fuel - FUEL_RESERVE) // FUEL_PER_ROTATION)
        row_paras, row_charge, total_paras, total_charge = [], [], 0, 0
        for i in range(max_rotations):
            if i < nb_rot:
                fuel_at_rot = fuel - i * FUEL_PER_ROTATION
                n = max_paras_for_fuel(fuel_at_rot)
                _, mass0, _ = compute_moment(
                    [], EW_KG, EW_MOMENT, POIDS_PARA_KG, POIDS_PILOTE_KG, fuel_at_rot)
                charge_kg = round(MTOW_KG - mass0)
                total_paras += n
                total_charge += charge_kg
                row_paras.append(str(n))
                row_charge.append(str(charge_kg))
            else:
                row_paras.append(None)
                row_charge.append(None)
        all_rows.append((fuel, nb_rot, row_paras, row_charge, total_paras, total_charge))

    return all_rows, max_rotations


def _draw_triangular_table(ax, table_data, header_bg, alt_color, color_total, subtitle, fs=12, has_total=True):
    """
    Tableau triangulaire : cellules None invisibles + barre verticale de fermeture
    a droite de la derniere cellule reelle (avant la colonne Total).
    """
    from matplotlib.patches import Rectangle as _Rect
    from matplotlib.lines import Line2D

    display_data = [['' if v is None else str(v) for v in row] for row in table_data]
    ncols = len(table_data[0])
    nrows = len(table_data)

    ax.axis('off')
    the_table = ax.table(cellText=display_data, loc='center', cellLoc='center',
                         bbox=[0, 0, 1, 1])
    the_table.auto_set_font_size(False)
    the_table.set_zorder(2)

    # last_real_col = derniere colonne non-None, hors colonne Total (ncols-1)
    last_real_col = {}
    for row in range(1, nrows):
        last_col = 0
        for col in range(ncols - 1 if has_total else ncols):
            if table_data[row][col] is not None:
                last_col = col
        last_real_col[row] = last_col

    cells = the_table.get_celld()

    for (row, col), cell in cells.items():
        if row >= nrows or col >= ncols:
            cell.set_visible(False)
            continue

        val = table_data[row][col]

        if val is None:
            cell.set_facecolor('white')
            cell.set_edgecolor('white')
            cell.get_text().set_text('')
            continue

        cell.set_linewidth(0.6)
        cell.set_edgecolor('#AAAAAA')

        if row == 0:
            cell.set_facecolor(header_bg)
            cell.get_text().set_color('white')
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs)
            cell.get_text().set_va('center')
        elif has_total and col == ncols - 1:
            cell.set_facecolor(color_total)
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs + 2)
        elif col <= 1:
            cell.set_facecolor('#E4EBF5')
            cell.get_text().set_fontweight('bold')
            cell.get_text().set_fontsize(fs + 1)
        else:
            cell.set_facecolor('#FFFFFF' if row % 2 == 0 else alt_color)
            cell.get_text().set_fontsize(fs + 1)

    # Barres de fermeture : verticale a droite de la derniere cellule reelle
    col_w = 1.0 / ncols
    row_h = 1.0 / nrows
    max_real_col = ncols - 2 if has_total else ncols - 1
    for row_idx in range(1, nrows):
        lv = last_real_col[row_idx]
        if lv < max_real_col:
            x_close = (lv + 1) * col_w
            y_bot = 1.0 - (row_idx + 1) * row_h
            y_top = 1.0 - row_idx * row_h

            # Barre verticale a droite de la derniere cellule reelle
            ax.add_artist(Line2D([x_close, x_close], [y_bot, y_top],
                                 transform=ax.transAxes,
                                 color='#AAAAAA', linewidth=0.6,
                                 zorder=100, clip_on=False,
                                 solid_capstyle='butt'))

            # Bas de la derniere cellule reelle (trait horizontal leger)
            x_left_cell = lv * col_w
            ax.add_artist(Line2D([x_left_cell, x_close], [y_bot, y_bot],
                                 transform=ax.transAxes,
                                 color='#AAAAAA', linewidth=0.6,
                                 zorder=100, clip_on=False,
                                 solid_capstyle='butt'))

    ax.set_title(subtitle, fontsize=fs + 2, pad=8, fontweight='bold')


def plot_rotations_page_pil(ax_top, ax_bot, all_rows, max_rotations, title_text, POIDS_PARA_KG, fs=12):
    """Genere les deux tableaux rotations (paras + masse kg) sur une meme page."""
    rot_headers = [f'Rot.{i+1}' for i in range(max_rotations)]

    # --- Table 1 : nombre de paras ---
    header_p = ['Carburant\n(L)', 'Nb\nrot.'] + rot_headers + ['Total\nparas']
    table_p = [header_p]
    for fuel, nb_rot, row_paras, _, total_paras, _ in all_rows:
        table_p.append([str(fuel), str(nb_rot)] + row_paras + [str(total_paras)])

    _draw_triangular_table(
        ax_top, table_p,
        header_bg='#2C5F8A', alt_color='#EAF2FB', color_total='#FFF3CD',
        subtitle=(f'{title_text}\n'
                  'Nombre de paras par rotation vs carburant embarque  '
                  '--  Conso: 50 L/rot  --  Reserve: 70 L (jour)'),
        fs=fs, has_total=True)

    # --- Table 2 : masse max embarquable (kg) = MTOW - EW - pilote - fuel ---
    header_c = ['Carburant\n(L)', 'Nb\nrot.'] + rot_headers + ['Total\ncharge\n(kg)']
    table_c = [header_c]
    for fuel, nb_rot, _, row_charge, _, total_charge in all_rows:
        table_c.append([str(fuel), str(nb_rot)] + row_charge + [str(total_charge)])

    _draw_triangular_table(
        ax_bot, table_c,
        header_bg='#1E6B3C', alt_color='#EAFAF1', color_total='#D5F5E3',
        subtitle='Masse max embarquable (kg) par rotation  --  MTOW - EW - pilote - carburant',
        fs=fs, has_total=True)


In [6]:
def get_slots_pil():
    #  ## TOP
    # x0_top = 2.9
    # x1_top = 5.3

    # nb_slots_top = 5
    # x_slots_top = [x0_top + i * (x1_top - x0_top) / (nb_slots_top - 1) for i in range(nb_slots_top)]
     ## TOP
    x0_top = 2.9
    x1_top = 4.4
    x2_top = 5.2

    nb_slots_top = 4
    x_slots_top = [x0_top + i * (x1_top - x0_top) / (nb_slots_top - 1) for i in range(nb_slots_top)] + [x2_top]

    ## BOTTOM
    x0_bot = 3.4
    x1_bot = 5.3

    nb_slots_bot = 5
    x_slots_bot = [x0_bot + i * (x1_bot - x0_bot) / (nb_slots_bot - 1) for i in range(nb_slots_bot)]

    return x_slots_top, x_slots_bot

In [7]:
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from datetime import date
import os

EW_KG = 1365
EW_MOMENT = 4635.12
POIDS_PILOTE_KG = 80
POIDS_PARA_KG = 90
MTOW_KG = 2800
DATUM_LINE = 3.0
MAC = 1.9
IMMAT = 'PC6-A'
VERSION = date.today().strftime('%d/%m/%Y')
CIVIL = False  # bascule a True dans les jumeaux civils (gen_civil_notebooks.py)
TARGET_CG_M = 3.66  # Milieu de la zone seulement limitee par la MTOW (3.608-3.722 m)

all_rows, max_rotations = _build_rotations_data_pil(
    MTOW_KG, DATUM_LINE, MAC, EW_KG, EW_MOMENT,
    POIDS_PARA_KG, POIDS_PILOTE_KG,
    target_cg_m=TARGET_CG_M)

version_fn = VERSION.replace('/', '-')
OUTPUT_DIR = os.path.join(os.pardir, 'output')  # notebooks/ -> output/
os.makedirs(OUTPUT_DIR, exist_ok=True)
filename = os.path.join(
    OUTPUT_DIR, f'Planches_centrage_{IMMAT}_90kg_{version_fn}.pdf')

with PdfPages(filename) as pdf:
    title_text = f'{IMMAT}: poids para: {POIDS_PARA_KG} (kg), poids pilote: {POIDS_PILOTE_KG} (kg)'

    # Page 1a/1b - Configuration : 2 pages de 5 mini-plans (2x3)
    n_values_all = [0] + list(range(2, 11))
    split = len(n_values_all) // 2 + (len(n_values_all) % 2)
    page_groups = [n_values_all[:split], n_values_all[split:]]
    for page_idx, n_vals in enumerate(page_groups):
        fig1, axes1 = plt.subplots(2, 3, figsize=(11.7*2, 8.3*2), dpi=200)
        suffix = f" (page {page_idx+1}/2)"
        fig1.suptitle(title_text + " - Placement des paras selon le nombre a bord" + suffix + "\n"
                      f"EW: {EW_KG} kg  |  EW Moment: {EW_MOMENT} kg.m  |  MTOW: {MTOW_KG} kg  |  Version: {VERSION}",
                      fontsize=20)
        plot_conf_grid_pil(axes1, 10, EW_KG, EW_MOMENT, MTOW_KG,
                           POIDS_PILOTE_KG, POIDS_PARA_KG,
                           n_values=n_vals, target_cg_m=TARGET_CG_M)
        fig1.tight_layout(rect=[0, 0, 1, 0.92])
        pdf.savefig(fig1, bbox_inches=_draw_profile_frame(fig1, CIVIL))
        plt.close(fig1)

    # Page 2 - Tableau masse et centrage
    fig4 = plt.figure(figsize=(11.7*2, 8.3*2), dpi=200)
    plt.title(title_text + " - Tableau masse & centrage", fontsize=30, pad=20)
    ax4 = fig4.gca()
    plot_data_table(ax4, MTOW_KG, DATUM_LINE, MAC, EW_KG, EW_MOMENT,
                    POIDS_PILOTE_KG, POIDS_PARA_KG,
                    target_cg_m=TARGET_CG_M)
    pdf.savefig(fig4, bbox_inches=_draw_profile_frame(fig4, CIVIL))
    plt.close(fig4)

    # Page 3 - Rotations
    fig5, (ax5_top, ax5_bot) = plt.subplots(
        2, 1, figsize=(11.7*2, 8.3*2), dpi=200,
        gridspec_kw={'hspace': 0.18, 'top': 0.97, 'bottom': 0.02})
    fig5.patch.set_facecolor('#FAFAFA')
    plot_rotations_page_pil(ax5_top, ax5_bot, all_rows, max_rotations, title_text, POIDS_PARA_KG, fs=12)
    pdf.savefig(fig5, bbox_inches=_draw_profile_frame(fig5, CIVIL))
    plt.close(fig5)


    # Annexe - Centrogramme (enveloppe + points critiques)
    fig_cp = plt.figure(figsize=(11.7*2, 8.3*2), dpi=200)
    ax_cp = fig_cp.gca()
    plot_envelope_critical_points_pil(ax_cp, EW_KG, EW_MOMENT, MTOW_KG,
                                       POIDS_PILOTE_KG, POIDS_PARA_KG,
                                       DATUM_LINE, MAC,
                                       target_cg_m=TARGET_CG_M, n_max=10)
    fig_cp.suptitle(title_text + " - Annexe : centrogramme & points critiques", fontsize=22)
    fig_cp.tight_layout(rect=[0, 0, 1, 0.95])
    pdf.savefig(fig_cp, bbox_inches=_draw_profile_frame(fig_cp, CIVIL))
    plt.close(fig_cp)

print(f"PDF genere : {filename} (5 pages)")


PDF genere : ../output/Planches_centrage_PC6-A_90kg_02-09-2026.pdf (5 pages)
